# 01 baseline MD — Li₆PS₅Cl 双势对比薄管道 (W5–W6)

未微调 baseline：同一 Li₆PS₅Cl 胞上跑 **MACE-MP-0** 与 **MatterSim** 两条 baseline，多温 NVT MD → σ(T) → Arrhenius 外推 300K → 对标实验 ~3.15 mS/cm。

**薄管道**：MD 跑短、统计未收敛，σ 只是数量级指示；**baseline** = 未微调，预期偏差 2–40%——*看见这个偏差正是目的*，微调是 W7。

**用法**：代码执行程序 → 更改运行时类型 → **T4 GPU** → 左侧 🔑 Secrets 建 `MP_API_KEY` → 全部运行。
重包（torch/mace/mattersim）在 Colab GPU 上；结构构建（01）与分析（03）本机也能跑。

In [ ]:
# 1) 装包 + 确认 GPU（mattersim 较大，首次约 3–5 分钟）
!pip install -q mace-torch mattersim kinisi pymatgen-analysis-diffusion mp-api ase pymatgen scipp
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('CUDA:', torch.cuda.is_available(), '| GPU:', gpu, '| device:', device)
# 若 False/CPU：更改运行时类型 → T4 GPU → 全部运行。
# 若 mattersim 装不上，先只跑 MACE（下面 02 加 --mlip mace），事后再补 MatterSim。

In [ ]:
# 2) 拿代码：A) 填 REPO_URL 用 git clone；B) 否则上传 ai4ssb-mlip-md.zip 自动解压
import os, glob, zipfile
REPO_URL = ''  # 例 'https://github.com/E1582271-dotcom/ai4ssb-mlip-md.git'
if REPO_URL:
    !git clone -q $REPO_URL p2 && echo cloned
    %cd p2
else:
    zf = None
    try:
        from google.colab import files
        print('请选择本机 ~/Desktop/ai4ssb-mlip-md.zip 上传 ...')
        up = files.upload()
        zf = sorted(up)[0]
    except Exception:
        zf = (glob.glob('ai4ssb-mlip-md*.zip') or glob.glob('*.zip') or [None])[0]
    if zf:
        with zipfile.ZipFile(zf) as z:
            z.extractall('.')       # 解压到当前目录，脚本直接就位
        print('解压完成:', zf)
    else:
        print('未找到 zip：把 ai4ssb-mlip-md.zip 拖到左侧 Files 面板再重跑本单元，或设 REPO_URL。')
print('cwd:', os.getcwd(), '| 脚本:', sorted(glob.glob('0*_*.py')))

In [ ]:
# 3) 从 Colab Secret 读 MP_API_KEY（给 01 拉真实 Li6PS5Cl 用）
import os
try:
    from google.colab import userdata
    os.environ['MP_API_KEY'] = userdata.get('MP_API_KEY')
    print('MP_API_KEY loaded from Colab Secret')
except Exception as e:
    print('设 MP_API_KEY 于 Colab Secrets（若 data/ 已有 config CIF 可跳过 01）:', e)

In [ ]:
# 4) 构建结构（CPU）：MP mp-985592 → S/Cl 无序枚举 → data/config*.cif
!python 01_build_structure.py

In [ ]:
# 5) baseline 多温 MD（薄：先 2 温点 30 ps 验证跑得通，两势）
# 首次 MACE / MatterSim 会各自下载权重。跑通后再加到 --temps 600,800,1000 --steps 50000。
!python 02_baseline_md.py --temps 600,1000 --steps 30000 --equilib 3000

In [ ]:
# 6) 分析：MSD → D（pymatgen + kinisi 误差棒）→ σ(T) → Arrhenius → 对标实验
!python 03_analyze_transport.py

In [ ]:
# 7) 展示图
import os
from IPython.display import Image, display
for f in ['figures/02_md_stability_mace.png', 'figures/02_md_stability_mattersim.png',
          'figures/03_arrhenius.png', 'figures/03_sigma300_vs_expt.png']:
    if os.path.exists(f):
        print(f); display(Image(f))

## 跑通之后

1. **加温点 + 加时长**：`--temps 600,800,1000 --steps 50000`（三点才能看 Arrhenius 是否线性）。
2. **加超胞**：本机 `01_build_structure.py --supercell 2,2,2`（416 原子）提升统计，但 T4 会慢。
3. **W6 口述 checkpoint**（自己过）：bcc 硫骨架 / Arrhenius 外推 / 为何要微调 / concerted migration。
4. **W8** 才租 AutoDL RTX 5090 跑收敛生产 MD（唯一付费点）。

薄管道的 σ 不要当真值：它验证“全流程跑得通”，不验证“σ 准”。跟实验的偏差 = baseline 诊断的素材。